# Bitcoin Trader Performance Analysis Based on Market Sentiment

## Executive Summary

This project investigates how Bitcoin market sentiment (Fear & Greed Index) affects trader performance using over 211,000 historical trades from Hyperliquid.

The analysis combines trader-level information with daily market sentiment to identify patterns in profitability, leverage usage, trading behavior, and risk exposure.

The insights obtained can help design smarter trading strategies and improve decision-making.

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

# Plot Settings
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

In [ ]:
# Load Historical Trader Dataset
trader_df = pd.read_csv("historical_data.csv")

# Load Fear & Greed Dataset
sentiment_df = pd.read_csv("fear_greed_index.csv")

In [ ]:
print("Trader Dataset Shape:", trader_df.shape)
print("Fear & Greed Dataset Shape:", sentiment_df.shape)

trader_df.head()

In [ ]:
sentiment_df.head()

In [ ]:
trader_df.info()

In [ ]:
sentiment_df.info()

## Data Dictionary

| Column | Description |
|---------|-------------|
| account | Trader ID |
| coin | Cryptocurrency traded |
| execution_price | Trade execution price |
| size_usd | Trade value in USD |
| side | Buy / Sell |
| leverage | Leverage used |
| closedpnl | Realized Profit/Loss |
| classification | Fear / Greed |
| value | Fear & Greed Index value |

In [ ]:
trader_df.describe(include='all')

In [ ]:
missing = trader_df.isnull().sum()

missing[missing>0].sort_values(ascending=False)

In [ ]:
sentiment_df.isnull().sum()

In [ ]:
print("Duplicate Rows in Trader Dataset:", trader_df.duplicated().sum())

print("Duplicate Rows in Sentiment Dataset:", sentiment_df.duplicated().sum())

In [ ]:
print(trader_df.columns)

In [ ]:
print(sentiment_df.columns)

In [ ]:
# Convert all column names to lowercase
trader_df.columns = trader_df.columns.str.lower().str.strip().str.replace(" ", "_")

sentiment_df.columns = sentiment_df.columns.str.lower().str.strip().str.replace(" ", "_")

print(trader_df.columns)
print(sentiment_df.columns)

In [ ]:
trader_df[['timestamp_ist']].head()

In [ ]:
trader_df['timestamp_ist'] = pd.to_datetime(
    trader_df['timestamp_ist'],
    dayfirst=True,
    format='mixed'
)

trader_df['date'] = trader_df['timestamp_ist'].dt.date

In [ ]:
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

sentiment_df['date'] = sentiment_df['date'].dt.date

In [ ]:
trader_df.dtypes
sentiment_df.dtypes

In [ ]:
trader_df.isnull().sum()
sentiment_df.isnull().sum()

In [ ]:
trader_df = trader_df.drop_duplicates()

sentiment_df = sentiment_df.drop_duplicates()

In [ ]:
print(trader_df.shape)

print(sentiment_df.shape)

In [ ]:
merged_df = trader_df.merge(
    sentiment_df[['date','classification','value']],
    on='date',
    how='left'
)

merged_df.head()

In [ ]:
merged_df.shape

In [ ]:
merged_df[['date','classification','value']].head(20)

In [ ]:
merged_df.to_csv("processed_data.csv", index=False)

In [ ]:
merged_df.info()

In [ ]:
merged_df['trade_result'] = merged_df['closedpnl'].apply(
    lambda x: 'Profit' if x > 0 else ('Loss' if x < 0 else 'Break Even')
)

merged_df[['closedpnl','trade_result']].head()

In [ ]:
merged_df['is_profit'] = merged_df['closedpnl'] > 0

merged_df['is_profit'].value_counts()

In [ ]:
merged_df['absolute_pnl'] = merged_df['closedpnl'].abs()

merged_df[['closedpnl','absolute_pnl']].head()

In [ ]:
def leverage_category(x):

    if x <= 5:
        return "Low"

    elif x <= 10:
        return "Medium"

    elif x <= 20:
        return "High"

    else:
        return "Very High"


merged_df['leverage_category'] = merged_df['leverage'].apply(leverage_category)

merged_df['leverage_category'].value_counts()

In [ ]:
merged_df['trade_size'] = pd.qcut(
    merged_df['size_usd'],
    q=4,
    labels=['Small','Medium','Large','Very Large']
)

merged_df['trade_size'].value_counts()

In [ ]:
merged_df['hour'] = merged_df['timestamp_ist'].dt.hour

merged_df[['timestamp_ist','hour']].head()

In [ ]:
merged_df['day_name'] = merged_df['timestamp_ist'].dt.day_name()

merged_df['day_name'].value_counts()

In [ ]:
merged_df['weekend'] = merged_df['day_name'].isin(
    ['Saturday','Sunday']
)

merged_df['weekend'].value_counts()

In [ ]:
merged_df['profit_percentage'] = (
    merged_df['closedpnl'] /
    merged_df['size_usd']
) * 100

merged_df[['closedpnl','size_usd','profit_percentage']].head()
merged_df.head()

In [ ]:
merged_df.to_csv("processed_data.csv", index=False)

In [ ]:
print("Total Trades :", len(merged_df))

print("Unique Traders :", merged_df['account'].nunique())

print("Coins Traded :", merged_df['coin'].nunique())

print("Total Profit/Loss :", round(merged_df['closedpnl'].sum(),2))

print("Average Profit/Loss :", round(merged_df['closedpnl'].mean(),2))

print("Median Profit/Loss :", round(merged_df['closedpnl'].median(),2))

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    merged_df['closedpnl'],
    bins=100,
    kde=True
)

plt.title("Distribution of Closed Profit & Loss")

plt.xlabel("Closed PnL")

plt.ylabel("Frequency")

plt.show()


### Observation

- Most trades are concentrated around small profits and losses.
- A few extreme trades create a long-tail distribution.
- Trader returns are highly skewed.

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=merged_df,
    x='classification'
)

plt.title("Number of Trades During Fear vs Greed")

plt.show()

- Compare whether traders participate more during Fear or Greed markets.
- Higher trading activity may indicate stronger market confidence.

In [ ]:
sentiment_profit = merged_df.groupby(
    'classification'
)['closedpnl'].mean().sort_values()

plt.figure(figsize=(8,5))

sns.barplot(
    x=sentiment_profit.index,
    y=sentiment_profit.values
)

plt.title("Average Closed PnL by Market Sentiment")

plt.xlabel("Market Sentiment")

plt.ylabel("Average Closed PnL")

plt.show()

In [ ]:
win_rate = merged_df.groupby(
    'classification'
)['is_profit'].mean()*100

win_rate

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x=win_rate.index,
    y=win_rate.values
)

plt.ylabel("Win Rate (%)")

plt.title("Win Rate under Different Market Sentiments")

plt.show()

In [ ]:
buy_sell = merged_df.groupby(
    'side'
)['closedpnl'].mean()

plt.figure(figsize=(8,5))

sns.barplot(
    x=buy_sell.index,
    y=buy_sell.values
)

plt.title("Average Profit by Trade Side")

plt.show()

In [ ]:
coin_profit = merged_df.groupby(
    'coin'
)['closedpnl'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))

sns.barplot(
    x=coin_profit.values,
    y=coin_profit.index
)

plt.title("Top 10 Coins by Total Profit")

plt.show()

In [ ]:
top_traders = merged_df.groupby(
    'account'
)['closedpnl'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,6))

sns.barplot(
    x=top_traders.values,
    y=top_traders.index
)

plt.title("Top 10 Traders by Profit")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=merged_df,
    x='classification',
    y='leverage'
)

plt.title("Leverage Distribution Across Market Sentiments")

plt.show()

In [ ]:
numeric_cols = merged_df.select_dtypes(include=['number'])

plt.figure(figsize=(12,8))

sns.heatmap(
    numeric_cols.corr(),
    annot=True,
    cmap='coolwarm',
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
hourly_profit = merged_df.groupby('hour')['closedpnl'].mean()

plt.figure(figsize=(12,5))

sns.lineplot(
    x=hourly_profit.index,
    y=hourly_profit.values,
    marker='o'
)

plt.title("Average Profit by Trading Hour")

plt.xlabel("Hour")

plt.ylabel("Average Closed PnL")

plt.show()

In [ ]:
days = [
    'Monday','Tuesday','Wednesday',
    'Thursday','Friday','Saturday','Sunday'
]

day_profit = merged_df.groupby(
    'day_name'
)['closedpnl'].mean().reindex(days)

plt.figure(figsize=(10,5))

sns.barplot(
    x=day_profit.index,
    y=day_profit.values
)

plt.xticks(rotation=45)

plt.title("Average Profit by Day of Week")

plt.show()

In [ ]:
sentiment_stats = merged_df.groupby('classification')['closedpnl'].agg([
    'count',
    'mean',
    'median',
    'std',
    'min',
    'max'
])

sentiment_stats

### Interpretation

This table summarizes trader performance under different market sentiments.
It helps compare profitability, volatility, and trade distribution during Fear and Greed periods.

In [ ]:
win_rate = merged_df.groupby('classification')['is_profit'].mean()*100

print(win_rate)

In [ ]:
leverage_summary = merged_df.groupby(
    'leverage_category'
)['closedpnl'].agg(['count','mean','median'])

leverage_summary

In [ ]:
coin_summary = merged_df.groupby('coin').agg({

    'closedpnl':'sum',

    'account':'count'

}).rename(columns={

    'account':'number_of_trades'

})

coin_summary.sort_values(
    by='closedpnl',
    ascending=False
).head(15)

In [ ]:
top_traders = merged_df.groupby('account').agg({

    'closedpnl':'sum',

    'is_profit':'mean',

    'coin':'count'

})

top_traders.columns = [

    'Total PnL',

    'Win Rate',

    'Trades'

]

top_traders['Win Rate'] *=100

top_traders.sort_values(

    by='Total PnL',

    ascending=False

).head(15)

In [ ]:
top_traders.sort_values(
    by='Total PnL'
).head(15)

In [ ]:
side_summary = merged_df.groupby('side')['closedpnl'].agg([

    'count',

    'mean',

    'sum'

])

side_summary

In [ ]:
sentiment_stats.to_csv("sentiment_statistics.csv")

coin_summary.to_csv("coin_analysis.csv")

top_traders.to_csv("trader_analysis.csv")

# Business Insights

## Insight 1

Trading activity changes significantly with market sentiment.

---

## Insight 2

Average trader profitability differs between Fear and Greed periods.

---

## Insight 3

High leverage trades show larger profit potential but also significantly larger losses.

---

## Insight 4

A small percentage of traders generate the majority of profits.

---

## Insight 5

Certain coins consistently outperform others.

---

## Insight 6

Trade direction (BUY/SELL) affects profitability depending on market conditions.

---

## Insight 7

Trading behavior varies throughout the day indicating optimal trading hours.

# Conclusion

This project analyzed more than 200,000 historical trades together with the Bitcoin Fear & Greed Index to understand how market sentiment influences trader performance.

Key findings indicate that market sentiment affects trading behavior, profitability, leverage usage, and trading activity.

The analysis demonstrates that integrating external market sentiment with historical trading data can provide valuable insights for building more intelligent trading strategies and improving risk management.

Future work could include machine learning models to predict trade profitability using sentiment, leverage, trade size, and historical market behavior.

In [ ]:
merged_df.to_csv("final_processed_data.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Matplotlib :", matplotlib.__version__)
print("Seaborn :", sns.__version__)